In [1]:
import time
import pyvisa
import numpy as np
import os
import re
import pandas as pd
import plotly.express as px

In [2]:
OSC_ADDRESS =  "TCPIP0::192.168.1.200::INSTR"

In [3]:
_osc = None


def open_oscilloscope(address=None, max_attempts=5, retry_delay=1):
    """Open a VISA connection to the oscilloscope and cache it on this module. Retries if connection fails."""
    global _osc
    # Use the pyvisa-py backend for consistent LAN/USB detection
    rm = pyvisa.ResourceManager('@py')
    target = address or OSC_ADDRESS
    for attempt in range(max_attempts):
        try:
            _osc = rm.open_resource(target, timeout=5000)  # 5 seconds, adjust as needed
            return _osc
        except Exception as e:
            print(f"Attempt {attempt+1} to open oscilloscope failed: {e}")
            time.sleep(retry_delay)
    print(f"Error: Could not open oscilloscope after {max_attempts} attempts.")
    return None

def close_oscilloscope():
    """Close the cached oscilloscope handle."""
    global _osc
    if _osc is not None:
        try:
            _osc.close()
        except Exception:
            pass
        _osc = None

In [4]:
def configure_oscilloscope_for_burst(osc, params):
    """Configure oscilloscope acquisition settings for burst waveform capture."""

    def vbs(cmd):
        osc.write(f"VBS '{cmd}'")
        time.sleep(0.1)

    freq = params["frequency"]
    amp = params["amplitude"]
    n_cycles = params["no_of_cycles_per_pulse"]

    burst_duration = n_cycles / freq
    hor_scale = burst_duration
    ver_scale = amp
    sampling_rate = freq * 10

    vbs(f"app.Acquisition.Horizontal.HorScale = {hor_scale}")
    vbs(f"app.Acquisition.C1.VerScale = {ver_scale}")
    vbs(f"app.Acquisition.Horizontal.SampleRate = {sampling_rate}")
    vbs("app.Acquisition.C1.Offset = 0")
    vbs("app.Acquisition.C1.View = true")
    vbs('app.Acquisition.Trigger.Source = "C1"')
    osc.write("TRIG_MODE NORM")

In [5]:
def _get_num(osc, cmd):
    """Helper to extract the first numeric value from a scope response."""
    resp = osc.query(cmd).strip()
    match = re.search(r"[-+]?\d*\.\d+|\d+E[-+]?\d+|\d+", resp, re.IGNORECASE)
    return float(match.group()) if match else 0.0

In [6]:
def read_oscilloscope_and_save(osc, SIGNAL_PARAMS, cross, s, scan_folder):
    osc = osc or _osc
    if not osc:
        print("Oscilloscope not open.")
        return

    try:
        # 1. Setup & Acquisition
        # configure_oscilloscope_for_burst(osc, SIGNAL_PARAMS)
        osc.write("COMM_HEADER OFF")
        osc.write("COMM_FORMAT DEF9,WORD,BIN")
        
        # Pull raw data first (The core of the capture)
        raw_data = osc.query_binary_values("C1:WF? DAT1", datatype="h", container=np.array)
        num_points = len(raw_data)

        # 2. Vertical Scaling (HDO 16-bit logic)
        v_div = _get_num(osc, "C1:VDIV?")
        v_off = _get_num(osc, "C1:OFST?")
        voltages = (raw_data * (v_div * 8 / 65536)) - v_off

        # 3. Horizontal Scaling
        t_div = _get_num(osc, "TDIV?")
        h_off = _get_num(osc, "TRDL?")  # Trigger Delay
        time_span = t_div * 10
        time_values = np.linspace(h_off, h_off + time_span, num_points, endpoint=False)

        # 4. Save Waveform
        df = pd.DataFrame({"Time (s)": time_values, "Amplitude (V)": voltages})
        # file_path = os.path.join(scan_folder, f"row_{cross + 1}_col_{s}.csv")
        # df.to_csv(file_path, index=False)

        # # 5. Save Parameters (Only if it's the first save, to avoid redundant writes)
        # param_path = os.path.join(scan_folder, "wave_parameter.csv")
        # if not os.path.exists(param_path):
        #     pd.DataFrame(list(SIGNAL_PARAMS.items()), columns=["Parameter", "Value"]).to_csv(param_path, index=False)

        # print(f"✅ Saved row_{cross+1}_col_{s} ({num_points} pts)")
        return df

    except Exception as e:
        print(f"❌ Failed to read data: {e}")
        return None


In [7]:
osc = None

try:
    osc = open_oscilloscope(OSC_ADDRESS)
except Exception:
    print("Error openning scope")
     

In [8]:
osc

<'TCPIPInstrument'('TCPIP0::192.168.1.200::inst0::INSTR')>

In [9]:
params = dict(
    frequency=5e6,
    amplitude=2.0,
    no_of_cycles_per_pulse=1,
    prf=10,
    total_acquisition_time=1.0,
 )

In [10]:
# configure_oscilloscope_for_burst(_osc, params)

In [11]:
# Run one read per pulse using PRF timing for total acquisition duration
if osc is None:
    print("Oscilloscope connection not established. Please run the connection cell first.")
else:
    osc.write('TRMD NORM')  # Normal/continuous trigger mode
    trigger_source = osc.query("VBS? 'return = app.Acquisition.Trigger.Source'").strip()
    print(f"Current trigger source: {trigger_source}")

    # Expected params keys: prf (Hz) and total_acquisition_time (seconds)
    prf_hz = params.get("prf", params.get("PRF"))
    total_acq_s = params.get("total_acquisition_time", params.get("total_acquisition_time_s", 1.0))

    if prf_hz is None or prf_hz <= 0:
        print("Please set params['prf'] to a value > 0 (Hz).")
    elif total_acq_s <= 0:
        print("Please set total acquisition time to a value > 0 seconds.")
    else:
        pulse_interval_s = 1.0 / float(prf_hz)
        num_triggers = int(float(prf_hz) * float(total_acq_s))
        start_time = time.time()

        print(f"PRF: {prf_hz} Hz, pulse interval: {pulse_interval_s:.6f} s")
        print(f"Total acquisition: {total_acq_s} s, captures: {num_triggers}")

        for i in range(num_triggers):
            target_time = start_time + (i * pulse_interval_s)
            while True:
                now = time.time()
                if now >= target_time:
                    break
                time.sleep(min(0.001, target_time - now))

            print(f"Capturing pulse {i + 1}/{num_triggers}...")
            df = read_oscilloscope_and_save(osc, params, cross=0, s=i, scan_folder="./data/")
            if df is not None:
                print(f"Data for pulse {i + 1} saved.")
            else:
                print(f"No data for pulse {i + 1}.")

Current trigger source: C1
PRF: 10 Hz, pulse interval: 0.100000 s
Total acquisition: 1.0 s, captures: 10
Capturing pulse 1/10...
Data for pulse 1 saved.
Capturing pulse 2/10...
Data for pulse 2 saved.
Capturing pulse 3/10...
Data for pulse 3 saved.
Capturing pulse 4/10...
Data for pulse 4 saved.
Capturing pulse 5/10...
Data for pulse 5 saved.
Capturing pulse 6/10...
Data for pulse 6 saved.
Capturing pulse 7/10...
Data for pulse 7 saved.
Capturing pulse 8/10...
Data for pulse 8 saved.
Capturing pulse 9/10...
Data for pulse 9 saved.
Capturing pulse 10/10...
Data for pulse 10 saved.


In [12]:
fig = px.line(df, x='Time (s)', y='Amplitude (V)', title='LeCroy HDO6054 Signal')
fig.show()

In [13]:
close_oscilloscope()

In [14]:
# configure_oscilloscope_for_burst(osc, params)

# cross=0 
# s=0
# scan_folder="./data/"

# osc.write("C1:WF? DAT1")
# raw_data = osc.query_binary_values(
#     "C1:WF? DAT1", datatype="B", container=np.array
# )

# scale = float(osc.query("C1:VDIV?").strip().split(" ")[0])
# v_offset = float(osc.query("C1:OFST?").strip().split(" ")[0])
# scale1 = 1 / 30
# voltages = ((raw_data - 128) * scale + v_offset - 128) * scale1

# time_div = float(osc.query("TDIV?").strip().split(" ")[0])
# num_points = len(raw_data)
# time_span = 10 * time_div  # total span across 10 divisions
# time_values = np.linspace(0, time_span, num_points, endpoint=False)

# filename = f"row_{cross + 1}_col_{s}.csv"
# file_path = os.path.join(scan_folder, filename)
# df = pd.DataFrame({"Time (s)": time_values, "Amplitude (V)": voltages})
# df.to_csv(file_path, index=False)
# print(f"✅ Data saved to: {file_path}")

# # Save signal parameters alongside waveform data
# param_path = os.path.join(scan_folder, "wave_parameter.csv")
# param_df = pd.DataFrame(
#     list(params.items()), columns=["Parameter", "Value"]
# )
# param_df.to_csv(param_path, index=False)
# print(f"📝 Signal parameters saved to: {param_path}")